# CA1 – Data Collection for Apartment Price Prediction Using Machine Learning

**Module:** Machine Learning  
**Programme:** Higher Diploma in Data Analytics for Business  
**Student ID:** 2026328  
**Assignment:** CA1  
**Project:** Apartment Price Prediction in Mongolia  
**Data Source:** [Unegui.mn](https://www.unegui.mn/l-hdlh/l-hdlh-zarna/)  
**GitHub Repository:** https://github.com/Dashkaa1114/Machine-Learning-Sep26

## 1. Introduction

This project aims to predict apartment prices in Mongolia using Machine Learning techniques. The dataset is collected from Unegui.mn, a Mongolian online classified advertisements website.

The collected data includes apartment prices, locations, sizes, number of rooms, building years, apartment complex names, and publication dates.

## 2. Data Collection

This notebook collects apartment listing data from Unegui.mn using Python, Requests, and BeautifulSoup.

The extracted information is stored directly in a PostgreSQL database. The code also checks duplicate listings, supports incremental data collection, and allows the collection process to continue after an interruption.

## 3. Project Objective

The main objective is to prepare a real-world apartment dataset for price prediction.

The next stages of this project will include data cleaning, exploratory data analysis, data visualisation, feature preparation, and Machine Learning model training and evaluation.

**Target Variable:** `price_mnt` — Apartment asking price in Mongolian Tugrik (MNT).

**Database Table:** `public.apartment_listings_live`

## Step 1 — Import libraries

*import the tools that Python needs.*

- `requests`: download website pages.
- `BeautifulSoup`: read HTML.
- `re`: find text patterns.
- `SQLAlchemy`: save records to PostgreSQL.
- `datetime`: work with publication dates.

In [4]:
# Import libraries

import os
import re
import time
from datetime import datetime, timezone, timedelta
from pathlib import Path
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import display
from sqlalchemy import create_engine, MetaData, Table, text, func, or_
from sqlalchemy.engine import URL
from sqlalchemy.dialects.postgresql import insert

In [ ]:

BASE_URL = "https://www.unegui.mn"
CATEGORY_URL = BASE_URL + "/l-hdlh/l-hdlh-zarna/oron-suuts-zarna/"

PERMISSION_GRANTED = True  # Confirmed authorisation; follow its scope and limits.
RUN_COLLECTION = True      # Change to True to start real collection.
COLLECTION_MODE = "initial"  # "initial" = finish all pages, "daily" = check recent pages.
DAILY_PAGES = 5            # Checked only in daily mode; adjust to authorised scope.
START_PAGE = 1
END_PAGE = None       # All accessible pages for initial mode (bounded by HARD_PAGE_LIMIT).
MAX_LISTINGS = None       # None means no separate limit on detail pages.
# Reuse page checkpoints so completed pages are not repeated.
RESUME = True              # Initial: skip completed pages; revisit interrupted page.
# Do not request a detail page again when its ID is already stored during initial mode.
SKIP_SAVED_LISTINGS = True  # Initial: skip stored IDs; daily rechecks to detect changes.
REQUEST_DELAY_SECONDS = 5.0
REQUEST_TIMEOUT_SECONDS = 30
MAX_NO_NEW_PAGES = 3
HARD_PAGE_LIMIT = 2500    # Guard against an endless loop.
USER_AGENT = "AcademicDataResearch/1.0 (authorised educational research)"

# Optional date-only backfill from stored listing URLs (do not combine with collection).
# Optional mode for older records missing published_at.
RUN_DATE_BACKFILL = False
DATE_BACKFILL_LIMIT = 50

# Notebook inside 'notebooks/' → .env one folder above it.
# Find the folder that contains this notebook.
folder = Path.cwd().resolve()
project_dir = folder.parent if folder.name.lower() == "notebooks" else folder

# Stop early if a setting has an invalid value.
assert COLLECTION_MODE in {"initial", "daily"}
assert not (RUN_COLLECTION and RUN_DATE_BACKFILL), "Select collection OR date backfill, not both."
assert DAILY_PAGES >= 1
assert START_PAGE >= 1
assert END_PAGE is None or END_PAGE >= START_PAGE
assert REQUEST_DELAY_SECONDS > 0
assert HARD_PAGE_LIMIT >= START_PAGE

print("Project:", project_dir)
print("Pages:", START_PAGE, "to", END_PAGE or "last detected")

Project: D:\Development\Learning\Assignment-Sep26\Machine-Learning-Sep26\CA1-ML-Apartment-Price-Prediction
Pages: 1 to last detected


## Step 3 — Connect to PostgreSQL

*The password stays in `.env`.*

`engine` is the object used for database operations. A connection is opened only when a cell needs it.

**Database name:** My existing PostgreSQL database. This step does not make a separate database.

In [ ]:
# Load credentials and create the same PostgreSQL engine.
# Read My database settings from the .env file.
load_dotenv(project_dir / ".env", override=True)
needed = ["PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]
missing = [name for name in needed if not os.getenv(name)]
if missing:
    raise ValueError(f"Missing .env settings: {missing}")

# Create a connection URL without printing my password.
connection_url = URL.create(
    "postgresql+psycopg",
    username=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    host=os.getenv("PGHOST"),
    port=int(os.getenv("PGPORT")),
    database=os.getenv("PGDATABASE"),
)
# Use this engine to read and write PostgreSQL records.
engine = create_engine(connection_url, pool_pre_ping=True)
with engine.connect() as conn:
    print("Connected database:", conn.execute(text("SELECT current_database()")).scalar_one())

Connected database: unegui_apartments_data


## Step 4 — Make safe website requests

*The program checks website access rules before downloading pages.*

1. `check_robots()` reads the site's `robots.txt`.
2. `fetch_html(url)` checks the URL, waits between requests and reads HTML.
3. The program stops on blocked or rate-limited requests instead of trying to bypass them.

In [4]:
# Prepare a web session and an empty robots.txt parser.
session = requests.Session()
session.headers["User-Agent"] = USER_AGENT
robots = RobotFileParser()
last_request = 0.0

In [5]:
# Check website access rules before crawling.
def check_robots():
    if not PERMISSION_GRANTED:
        raise PermissionError("Collection is disabled; permission is required.")
    response = session.get(BASE_URL + "/robots.txt", timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    robots.parse(response.text.splitlines())

In [ ]:
# Request one HTML page; 
def fetch_html(url):
    global last_request
    if not PERMISSION_GRANTED:
        raise PermissionError("Collection is disabled.")
    # Only use HTTPS pages from the selected site.
    parsed = urlparse(url)
    if parsed.scheme != "https" or parsed.netloc != "www.unegui.mn":
        raise ValueError("Only Unegui.mn HTTPS pages are supported.")
    if not robots.can_fetch(USER_AGENT, url):
        raise PermissionError(f"Blocked by robots.txt: {url}")

    # Respect the minimum time between requests.
    # Wait when requests are too close together.
    if last_request:
        time.sleep(max(0, REQUEST_DELAY_SECONDS - (time.monotonic() - last_request)))
    response = session.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
    last_request = time.monotonic()
    # Stop if access is denied or the server requests a slower rate.
    if response.status_code in (403, 429):
        raise RuntimeError(f"Site rejected or rate-limited requests: {response.status_code}")
    response.raise_for_status()
    if "text/html" not in response.headers.get("Content-Type", "").lower():
        raise ValueError("The response was not HTML.")
    # Convert downloaded HTML into objects we can search.
    return BeautifulSoup(response.text, "html.parser")

## Step 5 — Find listing links

*We open a category page and find `/adv/` links. Each listing has an ID.*

`get_listing_urls()` returns unique URLs in page order. `listing_id_from_url()` reads the number from one URL.

In [7]:
# Keep the Original listing URL and ID rules.
LINK_RE = re.compile(r"^/adv/(\d+)(?:[_/]|$)")


def get_listing_urls(page_soup):
    urls = []
    # Look at every link on the category page.
    for a in page_soup.select("a[href]"):
        path = urlparse(urljoin(BASE_URL, a["href"])).path
        # Keep only links that look like Unegui advert pages.
        if LINK_RE.match(path):
            link = urljoin(BASE_URL, path)
            if link not in urls:
                urls.append(link)
    return urls


def listing_id_from_url(url):
    match = LINK_RE.match(urlparse(url).path)
    return match.group(1) if match else None

## Step 6 — Read apartment features

*HTML may contain a list of feature labels and values.*

`extract_features()` reads the Mongolian labels. `number()` converts a text number such as `41.42 м²` into a Python number.

In [8]:
# Find feature labels and convert text to numbers.
def extract_features(soup):
    section = soup.select_one('[data-component="AdvertFeaturesApp"]')
    if section is None:
        return {}
    values = {}
    # Read each feature row from the HTML.
    for block in section.select("div"):
        label = block.find("dt", recursive=False)
        value = block.find("dd", recursive=False)
        if label and value:
            values[label.get_text(" ", strip=True).rstrip(": ")] = value.get_text(" ", strip=True)
    return values


def number(value, integer=False):
    # Find the numeric part of the text.
    match = re.search(r"\d+(?:[.,]\d+)?", str(value or ""))
    if not match:
        return None
    num = float(match.group().replace(",", "."))
    return int(num) if integer else num

### Step 6B — Use consistent column names

*We change website labels into English database column names.*

In [9]:
# Map Mongolian labels to the Original database field names.
FEATURE_NAMES = {
    "Шал": "floor_material", "Тагт": "balcony",
    "Ашиглалтанд орсон он": "building_year", "Ашиглалтад орсон он": "building_year",
    "Гараж": "garage", "Цонх": "window_type", "Барилгын давхар": "total_floors",
    "Хаалга": "door_type", "Талбай": "area_m2", "Хэдэн давхарт": "floor_number",
    "Төлбөрийн нөхцөл": "payment_terms", "Лизингээр авах боломж": "payment_terms",
    "Цонхны тоо": "window_count", "Барилгын явц": "construction_status",
    "Цахилгаан шаттай эсэх": "elevator",
}


def clean_features(raw):
    # Select recognised feature labels and rename them.
    result = {FEATURE_NAMES[key]: value for key, value in raw.items() if key in FEATURE_NAMES}
    result["area_m2"] = number(result.get("area_m2"))
    for key in ["building_year", "total_floors", "floor_number", "window_count"]:
        result[key] = number(result.get(key), integer=True)
    return result

## Step 7 — Find apartment-complex names

*First, check an explicit feature name. Next, inspect the title and description.*

The result has three parts: `complex_name`, `complex_source`, `complex_review_needed`. A missing or uncertain name can remain `None`.

In [10]:
# Prepare the same complex-name text pattern and location words.
COMPLEX_RE = re.compile(r"(?<!\w)([\w-]+(?:\s+[\w-]+){0,3})\s+хотхон(?:д|ы|оос|оор)?\b", re.I)
LOCATION_WORDS = {
    "худ", "бзд", "сбд", "бгд", "схд", "чд", "дүүрэг", "хороо",
    "хорооны", "байршил", "орчим", "хажууд", "ойролцоо", "ар", "ард", "урд", "хойно"
}

In [11]:
# Find possible complex names in one text value.
def possible_complexes(text_value):
    matches = []
    # Search for words before the Mongolian word "хотхон".
    for match in COMPLEX_RE.finditer(text_value or ""):
        words = match.group(1).split()
        # Remove district/area prefixes; retain the probable complex name.
        # Remove location words that appear before the name.
        for i, word in enumerate(words):
            if word.lower().strip(",.:") in LOCATION_WORDS:
                words = words[i + 1:]
        name = " ".join(words).strip(" -.,;:")
        if name and name.casefold() not in [x.casefold() for x in matches]:
            matches.append(name)
    return matches

In [12]:
# Choose a complex name using the Original priority and conflict rules.
def extract_complex_name(title, description, raw_features):
    # Try the explicit apartment feature first.
    for key in ["Хотхон", "Хотхоны нэр", "Орон сууцны хотхон", "Төслийн нэр"]:
        if raw_features.get(key):
            return raw_features[key].strip(), "advert_feature", False

    # Search the title and description if there is no explicit feature.
    from_title = possible_complexes(title)
    from_description = possible_complexes(description)
    # Do not guess when the title has more than one possible name.
    if len(from_title) > 1:
        return None, "multiple_title_mentions", True
    if len(from_title) == 1:
        name = from_title[0]
        if from_description and name.casefold() not in [x.casefold() for x in from_description]:
            return None, "title_description_conflict", True
        source = "title_and_description" if from_description else "title"
        return name, source, True
    if len(from_description) == 1:
        return from_description[0], "description", True
    if len(from_description) > 1:
        return None, "multiple_description_mentions", True
    return None, "not_found", True

## Step 8 — Read the price and other text

*The website can show 183 сая ₮ or 1.03 тэрбум ₮.*

`price_to_mnt()` changes these strings into numeric Mongolian tugrik values. Other helper functions read text, remove contact details and validate coordinates.

In [13]:
# Convert an advertised price into MNT; keep the Original conversion rules.
PRICE_RE = re.compile(r"(\d+(?:[.,]\d+)?)\s*(тэрбум|сая)\s*₮", re.I)


def price_to_mnt(raw_value):
    value = str(raw_value or "")
    match = PRICE_RE.search(value)
    if match:
        multiplier = 1_000_000_000 if match.group(2).lower() == "тэрбум" else 1_000_000
        return round(float(match.group(1).replace(",", ".")) * multiplier)
    plain = re.fullmatch(r"\s*(\d{5,15})(?:\.0+)?\s*", value.replace(",", ""))
    return int(plain.group(1)) if plain else None

In [14]:
# Read text from an HTML element or attribute.
def get_text(soup, selectors):
    for selector in selectors:
        tag = soup.select_one(selector)
        if tag:
            value = tag.get("content") or tag.get("datetime") or tag.get_text(" ", strip=True)
            if value:
                return str(value).strip()
    return None


def get_attr(soup, pairs):
    for selector, attr in pairs:
        tag = soup.select_one(selector)
        if tag and tag.get(attr):
            return str(tag[attr]).strip()
    return None

In [15]:
# Remove contact information and validate latitude/longitude values.
def redact_contacts(value):
    if not value:
        return None
    value = re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", "[email removed]", value)
    return re.sub(r"(?<!\d)(?:\+?976[- ]?)?\d{4}[- ]?\d{4}(?!\d)", "[phone removed]", value)


def safe_coordinate(value, bound):
    try:
        value = float(str(value).replace(",", "."))
        return value if -bound <= value <= bound else None
    except (ValueError, TypeError):
        return None

## Step 9 — Find the publication date

*The website may say `Өнөөдөр` (today), `Өчигдөр` (yesterday), or show a full date.*

We interpret relative dates using Mongolia's UTC+08 day. The number in `Өчигдөр 6 ID: ...` is **not** treated as an hour. Only the calendar date is stored in `published_at`.

In [16]:
# These patterns recognise the Original date formats.
ULAANBAATAR_TZ = timezone(timedelta(hours=8))
DATE_TOKEN = r"(?:\d{4}[-./]\d{1,2}[-./]\d{1,2}|Өнөөдөр|Өчигдөр)"
POSTED_LABEL_RE = re.compile(
    rf"Нийтэлсэн\s*:\s*(?P<date>{DATE_TOKEN})(?:\s+\d{{1,2}}:\d{{2}})?",
    re.I,
)
# New site layout: "Өчигдөр 6 ID: 10836981" (6 is a counter, not an hour).
POSTED_INLINE_RE = re.compile(
    rf"(?P<date>{DATE_TOKEN})(?:\s+\d{{1,2}}:\d{{2}})?"
    rf"\s+(?:\d+\s+)?ID\s*:\s*\d+",
    re.I,
)

In [17]:
# Convert a website date label into a Python date.
def parse_posted_date(raw, fetched_at):
    """Return a Python date, or None if the calendar date cannot be determined."""
    if not raw:
        return None
    token = str(raw).strip()
    local_today = fetched_at.astimezone(ULAANBAATAR_TZ).date()
    if token.lower() == "өнөөдөр":
        return local_today
    if token.lower() == "өчигдөр":
        return local_today - timedelta(days=1)
    if re.fullmatch(r"\d{4}[-./]\d{1,2}[-./]\d{1,2}", token):
        try:
            return datetime(*[int(n) for n in re.split(r"[-./]", token)]).date()
        except ValueError:
            return None
    return None

In [18]:
# Find the date in visible text or structured HTML metadata.
def extract_published_date(soup, fetched_at):
    """Extract the calendar date displayed by Unegui, never invent an hour."""
    content = soup.get_text(" ", strip=True)
    found = POSTED_LABEL_RE.search(content) or POSTED_INLINE_RE.search(content)
    if found:
        return parse_posted_date(found.group("date"), fetched_at)

    # Fallback: use explicitly marked structured publication times if present.
    structured = get_attr(soup, [
        ('time[datetime]', 'datetime'),
        ('meta[itemprop="datePosted"]', 'content'),
        ('meta[property="article:published_time"]', 'content'),
    ])
    if structured:
        try:
            parsed = datetime.fromisoformat(structured.replace("Z", "+00:00"))
            if parsed.tzinfo is None:
                parsed = parsed.replace(tzinfo=ULAANBAATAR_TZ)
            return parsed.astimezone(ULAANBAATAR_TZ).date()
        except ValueError:
            return None
    return None

## Step 10 — Parse one complete apartment page

*This function combines the work from earlier steps into one dictionary.*

Example keys: `listing_id`, `title`, `price_mnt`, `rooms`, `area_m2`, `building_year`, `complex_name`, `published_at`, and `collected_at`.

The full list of fields, search order, and fallback rules are preserved. Some lines are long because changing these rules could change results.

In [19]:
# Combine all extracted fields into one apartment record.
def parse_listing(soup, listing_url):
    # Record the moment this page is processed.
    fetched_at = datetime.now(timezone.utc)
    published_at = extract_published_date(soup, fetched_at)
    doc_title = soup.title.get_text(" ", strip=True) if soup.title else ""
    title = get_text(soup, ["h1"]) or PRICE_RE.split(doc_title, maxsplit=1)[0].strip() or None
    title = redact_contacts(title)

    price_text = get_text(soup, ['[itemprop="price"]', '[data-testid="price"]'])
    if not price_to_mnt(price_text):
        price_text = get_attr(soup, [
            ('meta[itemprop="price"]', "content"),
            ('meta[property="product:price:amount"]', "content"),
        ]) or price_text
    if not price_to_mnt(price_text):
        match = PRICE_RE.search(doc_title)
        if match:
            price_text = match.group(0)

    region = get_text(soup, [
        '[itemprop="address"]', '[itemprop="addressRegion"]', '[itemprop="addressLocality"]'
    ])
    if region is None:
        match = re.search(r"\bin\s+(.+?)\s+-\s+\d+\s*өрөө", doc_title)
        region = match.group(1).strip() if match else None

    description = redact_contacts(get_text(soup, [
        '[data-component="AdvertDescriptionApp"]', '[itemprop="description"]',
        '.announcement-description', '#description'
    ]))
    # Read the property features from the HTML.
    raw_features = extract_features(soup)
    complex_name, complex_source, review = extract_complex_name(title, description, raw_features)
    room_match = re.search(r"(?<!\d)(\d+)\s*өрөө", title or doc_title)

    price = price_to_mnt(price_text)
    # Create one Python dictionary for one apartment.
    record = {
        "listing_id": listing_id_from_url(listing_url),
        "listing_url": listing_url,
        "title": title,
        "price_text": price_text,
        "price_mnt": price,
        "currency": get_attr(soup, [
            ('meta[itemprop="priceCurrency"]', "content"),
            ('meta[property="product:price:currency"]', "content")
        ]) or ("MNT" if price is not None else None),
        "rooms": int(room_match.group(1)) if room_match else None,
        "city_region": region,
        "complex_name": complex_name,
        "complex_source": complex_source,
        "complex_review_needed": review,
        "published_at": published_at,          # DATE only; no invented hour
        "description": description,
        "latitude": safe_coordinate(get_attr(soup, [
            ('[data-default-lat]', "data-default-lat"), ('[data-lat]', "data-lat"),
            ('meta[itemprop="latitude"]', "content")
        ]), 90),
        "longitude": safe_coordinate(get_attr(soup, [
            ('[data-default-lng]', "data-default-lng"), ('[data-lng]', "data-lng"),
            ('meta[itemprop="longitude"]', "content")
        ]), 180),
        "collected_at": fetched_at,
    }
    # Add the cleaned building features to the record.
    record.update(clean_features(raw_features))
    return record

## Step 11 — Prepare the real database tables

*The Original creates tables if they do not exist. It can also add missing columns and convert older `published_at` columns to DATE.*

**Warning:** This block runs real SQL, even when `RUN_COLLECTION = False`. Use it only on a database I have backed up. We keep it identical because I asked for the same Original behaviour.

In [20]:
# Run the Original schema-setup SQL, then read the table structure.
# Begin a database transaction for table creation and schema updates.
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS public.apartment_listings_live (
            listing_id TEXT PRIMARY KEY, listing_url TEXT, title TEXT,
            price_text TEXT, price_mnt BIGINT, currency TEXT, rooms INTEGER,
            city_region TEXT, complex_name TEXT, complex_source TEXT,
            complex_review_needed BOOLEAN, published_at DATE,
            description TEXT, latitude DOUBLE PRECISION, longitude DOUBLE PRECISION,
            floor_material TEXT, balcony TEXT, building_year INTEGER, garage TEXT,
            window_type TEXT, total_floors INTEGER, door_type TEXT,
            area_m2 DOUBLE PRECISION, floor_number INTEGER, payment_terms TEXT,
            window_count INTEGER, construction_status TEXT, elevator TEXT,
            collected_at TIMESTAMPTZ
        )
    """))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS public.apartment_crawl_progress (
            page_number INTEGER PRIMARY KEY, listing_url_count INTEGER NOT NULL,
            saved_count INTEGER NOT NULL, skipped_existing_count INTEGER NOT NULL,
            completed_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        )
    """))
    # Only add missing columns; do not delete existing listing records.
    # If an older version of the table already exists, add just the new fields.
    for col, data_type in [
        ("complex_name", "TEXT"),
        ("complex_source", "TEXT"),
        ("complex_review_needed", "BOOLEAN"),
        ("published_at", "DATE"),
    ]:
        conn.execute(text(f"ALTER TABLE public.apartment_listings_live ADD COLUMN IF NOT EXISTS {col} {data_type}"))

    # Earlier notebook versions created published_at as TIMESTAMPTZ.
    # Convert to DATE without changing other property fields or deleting records.
    stored_type = conn.execute(text("""
        SELECT data_type FROM information_schema.columns
        WHERE table_schema = 'public'
          AND table_name = 'apartment_listings_live'
          AND column_name = 'published_at'
    """)).scalar_one()
    if stored_type == "timestamp with time zone":
        conn.execute(text("""
            ALTER TABLE public.apartment_listings_live
            ALTER COLUMN published_at TYPE DATE
            USING (published_at AT TIME ZONE 'Asia/Ulaanbaatar')::date
        """))
        print("Converted published_at from TIMESTAMPTZ to DATE (Mongolia local date).")
    elif stored_type == "timestamp without time zone":
        conn.execute(text("""
            ALTER TABLE public.apartment_listings_live
            ALTER COLUMN published_at TYPE DATE
            USING published_at::date
        """))
        print("Converted published_at from TIMESTAMP to DATE.")
    elif stored_type != "date":
        raise TypeError(f"Unexpected published_at PostgreSQL type: {stored_type}")

listing_table = Table("apartment_listings_live", MetaData(), schema="public", autoload_with=engine)
valid_columns = set(listing_table.columns.keys())
print("Tables ready.")

Tables ready.


## Step 12 — Save one listing (UPSERT)

*UPSERT means insert a new listing or update an existing one.*

- The listing ID prevents duplicate records.
- When the new page has `NULL`, keep an older saved field value.
- `collected_at` changes only if a meaningful stored value changes.
- The function returns `inserted`, `updated` or `unchanged`.

In [21]:
# Save a listing with the same Original SQL UPSERT behaviour.
def save_listing(record, was_stored=False):
    """Return 'inserted', 'updated' or 'unchanged'. Never create duplicate listing IDs."""
    # Keep only fields that exist in the database table.
    values = {key: value for key, value in record.items() if key in valid_columns}
    if not values.get("complex_name"):
        values.pop("complex_source", None)
        values.pop("complex_review_needed", None)
    if not values.get("listing_id"):
        raise ValueError("Cannot save without listing_id")

    # Prepare an INSERT statement for this listing.
    statement = insert(listing_table).values(**values)
    # Keep the old value if parsing the newer page gives us NULL.
    # Keep the old database value if the new parsed value is missing.
    new_values = {
        key: func.coalesce(getattr(statement.excluded, key), listing_table.c[key])
        for key in values if key not in {"listing_id", "collected_at"}
    }
    # Rechecking without a real property/date change does not update the row.
    # Check if a real database field has changed.
    changed = or_(*(
        listing_table.c[key].is_distinct_from(value)
        for key, value in new_values.items()
    ))
    update_fields = dict(new_values)
    update_fields["collected_at"] = statement.excluded.collected_at

    # Save in a database transaction.
    with engine.begin() as conn:
        result = conn.execute(statement.on_conflict_do_update(
            index_elements=["listing_id"],
            set_=update_fields,
            where=changed
        ))
    if result.rowcount == 0:
        return "unchanged"
    return "updated" if was_stored else "inserted"

## Step 13 — Remember saved listings and completed pages

*This is the RESUME feature.*

A page is only marked complete after the whole page succeeds. If the internet fails part-way through pages, that page can be checked again. Stored IDs are not added twice.

In [22]:
# Find completed page checkpoints and previously saved listing IDs.
def completed_pages():
    if not RESUME:
        return set()
    with engine.connect() as conn:
        return set(conn.execute(text("SELECT page_number FROM public.apartment_crawl_progress")).scalars())


def stored_ids():
    # Daily mode needs old IDs to identify updated vs newly inserted records.
    if COLLECTION_MODE == "initial" and not SKIP_SAVED_LISTINGS:
        return set()
    with engine.connect() as conn:
        return set(conn.execute(text("SELECT listing_id FROM public.apartment_listings_live")).scalars())

In [23]:
# Mark a page complete only after the original crawl loop finishes the page.
def mark_page_done(page, total_urls, saved, skipped):
    with engine.begin() as conn:
        conn.execute(text("""
            INSERT INTO public.apartment_crawl_progress
                (page_number, listing_url_count, saved_count, skipped_existing_count, completed_at)
            VALUES (:page, :urls, :saved, :skipped, NOW())
            ON CONFLICT (page_number) DO UPDATE SET
                listing_url_count = EXCLUDED.listing_url_count,
                saved_count = EXCLUDED.saved_count,
                skipped_existing_count = EXCLUDED.skipped_existing_count,
                completed_at = NOW()
        """), {"page": page, "urls": total_urls, "saved": saved, "skipped": skipped})

## Step 14 — Simple offline checks

*These small tests use example HTML, not the real website.*

They check the original price, date and apartment-field parsers.

In [24]:
# Run the same Original parser checks (no network requests).
from datetime import datetime, timezone
sample_time = datetime(2026, 10, 23, 8, 0, tzinfo=timezone.utc)
assert parse_posted_date("Өчигдөр", sample_time).isoformat() == "2026-10-22"
assert extract_published_date(
    BeautifulSoup("<div>Өчигдөр 6 ID: 10836981</div>", "html.parser"),
    sample_time
).isoformat() == "2026-10-22"
assert extract_published_date(
    BeautifulSoup("<div>Нийтэлсэн: 2026-10-18 10:30</div>", "html.parser"),
    sample_time
).isoformat() == "2026-10-18"
assert price_to_mnt("183 сая ₮") == 183_000_000
assert price_to_mnt("1.03 тэрбум ₮") == 1_030_000_000
assert listing_id_from_url("https://www.unegui.mn/adv/10836981_test/") == "10836981"
assert extract_features(BeautifulSoup("""<section data-component="AdvertFeaturesApp"><div><dt>Талбай</dt><dd>41.42 м²</dd></div><div><dt>Ашиглалтанд орсон он</dt><dd>2024</dd></div></section>""", "html.parser"))["Талбай"] == "41.42 м²"
print("Offline parser tests: PASS")

Offline parser tests: PASS


## Step 15 — Main collection loop

*This is the complete page-by-page crawler.*

1. Check website rules and load completed page numbers.
2. Visit category pages in order.
3. Read listing links and skip stored IDs in initial mode.
4. Parse and save each new listing immediately.
5. Stop on errors, keep the saved work, and resume later.
6. Mark only fully completed pages in the checkpoint table.

In [25]:
# Main crawl function; keep the Original decisions and progress logic.
def crawl_all_pages():
    if not PERMISSION_GRANTED:
        print("Collection disabled.")
        return
    # Check access rules before requesting any listing.
    check_robots()
    # Load pages that were already finished.
    done = completed_pages() if COLLECTION_MODE == "initial" else set()
    saved_ids = stored_ids()
    seen = set()
    pages_without_links = 0
    totals = {"inserted": 0, "updated": 0, "unchanged": 0, "skipped": 0}
    processed_this_run = 0

    # Daily check is always bounded; initial mode can follow pagination to its end.
    final_page = (min(HARD_PAGE_LIMIT, START_PAGE + DAILY_PAGES - 1)
                  if COLLECTION_MODE == "daily" else
                  (END_PAGE if END_PAGE is not None else HARD_PAGE_LIMIT))
    print(f"Mode: {COLLECTION_MODE}; pages {START_PAGE}–{final_page}")

    # Visit category pages one by one.
    for page in range(START_PAGE, final_page + 1):
        # Skip a page if the original initial run completed it.
        if page in done:
            print(f"Page {page}: completed earlier, skipping")
            continue

        page_url = CATEGORY_URL if page == 1 else f"{CATEGORY_URL}?page={page}"
        print(f"\nPage {page} | {page_url}")
        try:
            # Get all advert URLs shown on this category page.
            links = get_listing_urls(fetch_html(page_url))
        except requests.HTTPError as exc:
            if exc.response is not None and exc.response.status_code in (404, 410):
                print("Page does not exist; stopping.")
                break
            print("Category fetch failed:", exc)
            break
        except (requests.RequestException, ValueError, RuntimeError, PermissionError) as exc:
            print("Category fetch failed:", exc)
            break

        if not links:
            print("No links on this page; stopping (check whether HTML changed).")
            break

        fresh = []
        for link in links:
            listing_id = listing_id_from_url(link)
            if listing_id and listing_id not in seen:
                seen.add(listing_id)
                fresh.append((listing_id, link))

        pages_without_links = pages_without_links + 1 if not fresh else 0
        if pages_without_links >= MAX_NO_NEW_PAGES:
            print("Repeated pages without new IDs within this run; stopping.")
            break

        page_counts = {"inserted": 0, "updated": 0, "unchanged": 0, "skipped": 0}
        failed = False
        # Open one advert at a time.
        for listing_id, link in fresh:
            if (COLLECTION_MODE == "initial" and SKIP_SAVED_LISTINGS
                    and listing_id in saved_ids):
                page_counts["skipped"] += 1
                continue
            try:
                # Read and parse the advert HTML.
                record = parse_listing(fetch_html(link), link)
                if record["listing_id"] != listing_id:
                    raise ValueError("Listing ID mismatch")
                # Save the listing immediately to PostgreSQL.
                action = save_listing(record, was_stored=listing_id in saved_ids)
                page_counts[action] += 1
                processed_this_run += 1
                saved_ids.add(listing_id)
            except requests.HTTPError as exc:
                if exc.response is not None and exc.response.status_code in (404, 410):
                    print(f"  Listing {listing_id} is no longer available; skipping")
                    continue
                print(f"  Interrupted on listing {listing_id}: {exc}")
                failed = True
                break
            except Exception as exc:
                print(f"  Interrupted on listing {listing_id}: {type(exc).__name__}: {exc}")
                failed = True
                break
            # Stop at the requested test limit; do not mark a partial page as complete.
            if MAX_LISTINGS is not None and processed_this_run >= MAX_LISTINGS:
                print("Listing limit reached; page not checkpointed.")
                failed = True
                break

        for key, count in page_counts.items():
            totals[key] += count
        print(f"Page {page}: URLs={len(links)} | " +
              " | ".join(f"{k}={v}" for k, v in page_counts.items()))
        if failed:
            print("Page interrupted; safely rerun later.")
            break

        # A daily recheck should NOT turn historical page progress into new checkpoints.
        # Only the initial run writes page-completed checkpoints.
        if COLLECTION_MODE == "initial":
            mark_page_done(page, len(links),
                           page_counts["inserted"] + page_counts["updated"],
                           page_counts["skipped"] + page_counts["unchanged"])
    else:
        if COLLECTION_MODE == "initial" and END_PAGE is None:
            print("Reached HARD_PAGE_LIMIT; check pagination.")

    print("\nFinal result:", totals)
    if COLLECTION_MODE == "daily":
        print("Reminder: only listings on checked pages were inspected for edits.")

## Step 16 — Start or skip real collection
For the saved 70 pages, keep `START_PAGE = 1` and `RESUME = True`. The existing checkpoints decide where to continue.

In [ ]:
# Run the main crawl function if collection is enabled.
if RUN_COLLECTION:
    crawl_all_pages()
else:
    print("RUN_COLLECTION=False: full collection skipped.")

Mode: initial; pages 84–86
Page 84: completed earlier, skipping

Page 85 | https://www.unegui.mn/l-hdlh/l-hdlh-zarna/oron-suuts-zarna/?page=85
Page 85: URLs=60 | inserted=60 | updated=0 | unchanged=0 | skipped=0

Page 86 | https://www.unegui.mn/l-hdlh/l-hdlh-zarna/oron-suuts-zarna/?page=86
Page 86: URLs=60 | inserted=57 | updated=0 | unchanged=0 | skipped=0

Final result: {'inserted': 117, 'updated': 0, 'unchanged': 0, 'skipped': 0}


## Step 17 — Check my database results

*We read totals and recent records from PostgreSQL.*

This is a read-only report. It shows stored listing counts, publication dates and completed pages.

In [27]:
# Read the Original collection summary.
with engine.connect() as conn:
    summary = conn.execute(text("""
        SELECT COUNT(*) AS total_listings,
               COUNT(DISTINCT listing_id) AS unique_listings,
               COUNT(price_mnt) AS with_price,
               COUNT(area_m2) AS with_area,
               COUNT(complex_name) AS with_complex_name,
               COUNT(published_at) AS with_published_date,
               COUNT(*) FILTER (WHERE published_at IS NULL) AS missing_published_date
        FROM public.apartment_listings_live
    """)).mappings().one()
    latest = pd.read_sql(text("""
        SELECT listing_id, city_region, complex_name,
               price_mnt, area_m2, published_at, collected_at
        FROM public.apartment_listings_live
        ORDER BY collected_at DESC NULLS LAST
        LIMIT 10
    """), conn)
    checkpoints = pd.read_sql(text("""
        SELECT page_number, listing_url_count, saved_count, completed_at
        FROM public.apartment_crawl_progress
        ORDER BY page_number DESC LIMIT 10
    """), conn)
print("DATABASE SUMMARY:", dict(summary))
print("LATEST STORED LISTINGS:")
display(latest)
print("COMPLETED PAGES (initial collection):")
display(checkpoints)

DATABASE SUMMARY: {'total_listings': 3604, 'unique_listings': 3604, 'with_price': 3604, 'with_area': 3604, 'with_complex_name': 1285, 'with_published_date': 662, 'missing_published_date': 2942}
LATEST STORED LISTINGS:


,listing_id,city_region,complex_name,price_mnt,area_m2,published_at,collected_at
0,10834357,Сүхбаатар,NaN,843000000,111.00,None,2026-10-24 08:19:26.335171+00:00
1,10821325,NaN,NaN,300000000,100.00,None,2026-10-24 08:19:20.373321+00:00
2,10834342,Баянзүрх,NaN,430000000,67.28,None,2026-10-24 08:19:14.427399+00:00
3,10834367,Хан-Уул,NaN,700000000,93.30,None,2026-10-24 08:19:08.507438+00:00
4,10834365,Хан-Уул,NaN,455000000,94.00,None,2026-10-24 08:19:02.582927+00:00
5,10754875,Хан-Уул,NaN,1020000000,120.16,None,2026-10-24 08:18:56.443306+00:00
6,10834372,Хан-Уул,NaN,525000000,70.00,None,2026-10-24 08:18:50.090191+00:00
7,10777625,Хан-Уул,NaN,255000000,64.27,None,2026-10-24 08:18:44.150694+00:00
8,10777641,Хан-Уул,нисэх анар,146000000,45.66,None,2026-10-24 08:18:38.231488+00:00
9,10777675,Хан-Уул,оргил стар,545000000,119.70,None,2026-10-24 08:18:32.268951+00:00


COMPLETED PAGES (initial collection):


,page_number,listing_url_count,saved_count,completed_at
0,86,60,57,2026-10-24 08:19:26.377326+00:00
1,85,60,60,2026-10-24 08:13:35.947143+00:00
2,84,60,52,2026-10-24 08:00:17.430730+00:00
3,83,60,58,2026-10-24 07:54:56.019436+00:00
4,82,60,56,2026-10-24 07:48:59.444209+00:00
5,81,60,54,2026-10-24 07:43:15.341550+00:00
6,80,60,59,2026-10-24 07:37:43.245421+00:00
7,79,60,47,2026-10-24 07:31:39.134434+00:00
8,78,60,0,2026-10-24 07:26:46.511300+00:00
9,77,60,0,2026-10-24 07:26:38.027562+00:00
